In [26]:
import pandas as pd
import numpy as np

import sys
sys.path.append('/data_nfs/og86asub/netmap/netmap-evaluation/')
from src.methods.grnboost2.grnboost2_config import GRNBoost2Config
from src.data_simulation.data_simulation_config import DataSimulationConfig
from src.pipelines.utils import PipelineConfig
import os.path as op
import os
from compute_metrics import build_augmented_network
from src.utils import write_config


def calculate_recovered_edges(inferred_grn, gold_standard_grn, k_values):
    """
    Computes the percentage of recovered edges for increasing k top edges in an inferred GRN.

    Args:
        inferred_grn (str): Path to a CSV file for the inferred GRN.
                                 The file should have columns: 'regulator', 'target', 'score'.
        gold_standard_grn (str): Path to a CSV file for the gold standard GRN.
                                      The file should have columns: 'regulator', 'target'.
        k_values (list): A list of integers representing the number of top edges to consider.

    Returns:
        pd.DataFrame: A DataFrame with columns 'k' and 'percentage_recovered',
                      showing the recovery percentage for each k value.
    """
    # Sort inferred edges by score in descending order
    inferred_grn = inferred_grn.sort_values(by='importance', ascending=False).reset_index(drop=True)

    # Create a set of gold standard edges for efficient lookup
    gold_standard_edges = set(zip(gold_standard_grn['source'], gold_standard_grn['target']))
    total_gold_standard_edges = len(gold_standard_edges)

    # Filter k_values to not exceed the total number of inferred edges
    max_k = len(inferred_grn)

    # Initialize a list to store results
    results = []

    for k in k_values:
        top_k_inferred = inferred_grn.head(k)

        # Create a set of the top k inferred edges
        top_k_edges = set(zip(top_k_inferred['TF'], top_k_inferred['target']))

        # Find the intersection (recovered edges)
        recovered_edges = len(top_k_edges.intersection(gold_standard_edges))

        # Calculate percentage of recovered edges
        if total_gold_standard_edges > 0:
            percentage = (recovered_edges / total_gold_standard_edges)
        else:
            percentage = 0  # Avoid division by zero if gold standard is empty

        results.append({'n_top': k, 'percentage_recovered': percentage, 'tp': recovered_edges, 'gs_count': total_gold_standard_edges, 'pp': len(top_k_edges) })

    results = pd.DataFrame(results)
    return results


def compute_metric(net, nets, k_thresholds, per_target=True):
    recovery_rates = []
    for grn in net.grn.unique():
        for n in range(len(nets)):
            if per_target:
                recovery_rate = calculate_recovered_edges_per_target( net[net.grn == grn],nets[n], k_thresholds)
            else:
                recovery_rate = calculate_recovered_edges(net[net.grn == grn], nets[n], k_thresholds)
            if grn-1 == n:
                recovery_rate['type'] = 'on_target'
            else:
                recovery_rate['type'] = 'off_target'
            
            recovery_rate['grn'] = int(grn-1)
            recovery_rate['net'] = n
            recovery_rates.append(recovery_rate)
    recovery_rates = pd.concat(recovery_rates)

    return recovery_rates

def unify_group_labelling(adata, grn_adata, col_adata, col_grn_adata, return_mapping=False):
    """
    Adjust group labelling such that grn_adata has the same group label than the 
    corresponding column in adata based on the grn column.

    Returns the data objects and a score of the matching as the cost of the matching
    divided by the number of cells.
    
    """
    print(adata.shape)
    print(grn_adata.shape)
    cm = contingency_matrix(adata.obs[col_adata], grn_adata.obs[col_grn_adata])
    row_ind, col_ind = linear_sum_assignment(cm, maximize = True)
    
    names_ad = np.unique(adata.obs[col_adata])
    names_grn = np.unique(grn_adata.obs[col_grn_adata])
    mapping = {}
    reverse_mapping = {}
    for i in range(len(row_ind)):
        reverse_mapping[names_grn[col_ind[i]]] = names_ad[row_ind[i]]

    col_grn_adata_remapped = col_grn_adata + '_remap'
    if isinstance(np.unique(grn_adata.obs[col_grn_adata])[0], str):
        grn_adata.obs[col_grn_adata_remapped] = [reverse_mapping[a] for a in grn_adata.obs[col_grn_adata]]
    else:
        grn_adata.obs[col_grn_adata_remapped] = [reverse_mapping[int(a)] for a in grn_adata.obs[col_grn_adata]]

    grn_adata.obs[col_grn_adata_remapped] = pd.Categorical(grn_adata.obs[col_grn_adata_remapped])

    score = cm[row_ind, col_ind].sum()/adata.obs.shape[0]
    print(adata.shape)
    print(grn_adata.shape)
    if return_mapping:
        return adata, grn_adata, score, reverse_mapping
    else:
        return adata, grn_adata, score


In [2]:
import scanpy as sc
import time
from netmap.downstream.clustering import *

In [15]:
grn_adata = sc.read_h5ad('/data_nfs/og86asub/netmap/netmap-evaluation/results/netmap/config_6/config_noise/net_60_10082_net_64_11307_net_84_11226/grn_lrp.h5ad')
adata = sc.read_h5ad('/data_nfs/og86asub/netmap/netmap-evaluation/data/simulated_data/config_noise/net_60_10082_net_64_11307_net_84_11226/data.h5ad')
dataset_config = DataSimulationConfig.read_yaml('/data_nfs/og86asub/netmap/netmap-evaluation/results/configurations/data_simulation/config_noise/net_60_10082_net_64_11307_net_84_11226.config.yaml')

In [16]:
# read network files
nets = [pd.read_csv(op.join('/data_nfs/og86asub/netmap/netmap-evaluation/data/clustered_network/', filename), sep=dataset_config.separator) for filename in dataset_config.edgelist]

#off_net = [(op.basename(op.dirname(filename)), pd.read_csv(op.join(pipeline_config.clustered_network_dir, filename), sep=dataset_config.separator)) for filename in dataset_config.common_edges]

print(nets)

[     source  target
0      ATF2   DDIT3
1      ATF2    ATF4
2      ATF2    ATF3
3      ATF2   TRIB3
4      ATF2     TNF
..      ...     ...
99    LITAF  MAPK14
100   LITAF     TNF
101   LITAF   HSPA5
102  ZNF292     GH1
103   ZFP64     TNF

[104 rows x 2 columns],    source  target
0    ARNT    ESR2
1    ARNT   ABCB1
2    ARNT  CYP1B1
3    ARNT     SRC
4    ARNT    IRS2
..    ...     ...
66   ESR2  SCN10A
67   ESR2  FCGR3A
68   ESR2    GCSH
69  HOXC6   ABCB1
70  HOXC6     MME

[71 rows x 2 columns]]
